# TA-03: Prediksi Niat Pembelian Pengunjung E-Commerce dengan Regresi Logistik

**Mata Praktikum:** Pembelajaran Mesin  
**Dataset:** Online Shoppers Purchasing Intention (12.330 sesi pengunjung)  
**Target:** `Revenue` (1 = pengunjung membeli, 0 = tidak membeli)

Alur notebook:
1. Import library dan memuat data
2. Pengecekan kualitas data
3. EDA (grafik disimpan ke folder `images/`)
4. Preprocessing (duplikat, encoding, split, scaling, SMOTE)
5. Pemodelan: tanpa SMOTE vs dengan SMOTE
6. Evaluasi: Confusion Matrix, Precision, Recall, F1, ROC-AUC
7. Analisis threshold
8. Interpretasi koefisien model
9. Menyimpan model untuk aplikasi Flask

## 1. Import library dan memuat data

In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score, roc_curve,
                             precision_score, recall_score, f1_score,
                             accuracy_score)
from imblearn.over_sampling import SMOTE

sns.set_theme(style="whitegrid")

DATA_PATH = Path("../data/online_shoppers_intention.csv")
MODEL_DIR = Path("../model")
IMG_DIR = Path("../images")
MODEL_DIR.mkdir(exist_ok=True)
IMG_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Ukuran data:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

## 2. Pengecekan kualitas data

In [ ]:
print("Total missing value  :", df.isnull().sum().sum())
print("Jumlah baris duplikat:", df.duplicated().sum())

print("\nDistribusi target (jumlah):")
print(df["Revenue"].value_counts())
print("\nDistribusi target (proporsi):")
print(df["Revenue"].value_counts(normalize=True).round(3))

**Catatan untuk laporan:** tidak ada missing value, ada duplikat yang perlu dihapus, 
dan kelas target tidak seimbang (*class imbalance*), sehingga akurasi saja tidak cukup sebagai metrik.

## 3. Exploratory Data Analysis (EDA)

### 3.1 Distribusi target

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(x="Revenue", data=df, hue="Revenue", palette=["#5B6B70", "#1F8A70"], legend=False)
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom")
plt.title("Distribusi Pengunjung: Tidak Beli (False) vs Beli (True)")
plt.xlabel("Revenue")
plt.ylabel("Jumlah sesi")
plt.savefig(IMG_DIR / "01_distribusi_target.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.2 Distribusi fitur numerik utama

In [ ]:
num_view = ["ProductRelated", "ProductRelated_Duration", "BounceRates", "ExitRates", "PageValues"]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), num_view):
    sns.histplot(df[col], bins=40, ax=ax, color="#0F2E33")
    ax.set_title(col)
axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.savefig(IMG_DIR / "02_distribusi_fitur_numerik.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.3 Fitur numerik vs target

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, col in zip(axes, ["PageValues", "ExitRates", "ProductRelated_Duration"]):
    sns.boxplot(x="Revenue", y=col, data=df, ax=ax, showfliers=False,
                hue="Revenue", palette=["#5B6B70", "#1F8A70"], legend=False)
    ax.set_title(f"{col} berdasarkan Revenue")
plt.tight_layout()
plt.savefig(IMG_DIR / "03_boxplot_fitur_vs_target.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.4 Tingkat pembelian berdasarkan bulan, tipe pengunjung, dan akhir pekan

In [ ]:
month_order = ["Feb", "Mar", "May", "June", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
rate_month = (df.groupby("Month")["Revenue"].mean().reindex(month_order) * 100)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

rate_month.plot(kind="bar", ax=axes[0], color="#0F2E33")
axes[0].set_title("Tingkat pembelian per bulan (%)")
axes[0].set_ylabel("% sesi yang membeli")
axes[0].set_xlabel("")

(df.groupby("VisitorType")["Revenue"].mean() * 100).plot(kind="bar", ax=axes[1], color="#0F2E33")
axes[1].set_title("Tingkat pembelian per tipe pengunjung (%)")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=15)

(df.groupby("Weekend")["Revenue"].mean() * 100).plot(kind="bar", ax=axes[2], color="#0F2E33")
axes[2].set_title("Tingkat pembelian: hari kerja vs akhir pekan (%)")
axes[2].set_xlabel("Weekend")
axes[2].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig(IMG_DIR / "04_tingkat_pembelian_kategori.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.5 Korelasi antar fitur numerik

In [ ]:
corr_cols = ["Administrative", "Administrative_Duration", "Informational", "Informational_Duration",
             "ProductRelated", "ProductRelated_Duration", "BounceRates", "ExitRates",
             "PageValues", "SpecialDay"]
corr = df[corr_cols].assign(Revenue=df["Revenue"].astype(int)).corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, square=True)
plt.title("Heatmap Korelasi")
plt.savefig(IMG_DIR / "05_korelasi_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

**Catatan untuk laporan:** `BounceRates` dan `ExitRates` berkorelasi sangat tinggi (multikolinearitas), 
begitu juga durasi dengan jumlah halaman. Regresi Logistik masih bisa dipakai, tetapi koefisien 
fitur yang berkorelasi tinggi perlu diinterpretasi dengan hati-hati.

## 4. Preprocessing

Langkah-langkah:
1. Hapus baris duplikat
2. Ubah target dan `Weekend` menjadi 0/1
3. One-hot encoding untuk kolom kategorikal. `OperatingSystems`, `Browser`, `Region`, dan `TrafficType` 
   berisi angka tetapi hanya kode kategori, sehingga harus di-encode (bukan di-scale)
4. Split data **dulu**, baru scaling dan SMOTE hanya pada data latih (mencegah *data leakage*)

In [ ]:
df_clean = df.drop_duplicates().copy()
print("Setelah hapus duplikat:", df_clean.shape)

df_clean["Revenue"] = df_clean["Revenue"].astype(int)
df_clean["Weekend"] = df_clean["Weekend"].astype(int)

# Simpan daftar kategori untuk dropdown di aplikasi
MONTH_ORDER = ["Feb", "Mar", "May", "June", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
months_present = [m for m in MONTH_ORDER if m in df_clean["Month"].unique()]
months_present += sorted(set(df_clean["Month"].unique()) - set(months_present))
visitor_types = sorted(df_clean["VisitorType"].unique().tolist())

cat_cols = ["Month", "VisitorType", "OperatingSystems", "Browser", "Region", "TrafficType"]
df_model = pd.get_dummies(df_clean, columns=cat_cols, drop_first=True, dtype=int)
print("Ukuran setelah encoding:", df_model.shape)

num_cols = ["Administrative", "Administrative_Duration", "Informational", "Informational_Duration",
            "ProductRelated", "ProductRelated_Duration", "BounceRates", "ExitRates",
            "PageValues", "SpecialDay"]

In [ ]:
X = df_model.drop(columns="Revenue")
y = df_model["Revenue"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

print("Data latih :", X_train.shape, "| proporsi beli:", round(y_train.mean(), 3))
print("Data uji   :", X_test.shape, "| proporsi beli:", round(y_test.mean(), 3))

# Nilai median (data mentah) untuk mengisi fitur yang tidak ada di form aplikasi
defaults = {c: float(X_train[c].median()) for c in num_cols}

In [ ]:
scaler = StandardScaler()
X_train_s = X_train.copy()
X_test_s = X_test.copy()
X_train_s[num_cols] = scaler.fit_transform(X_train[num_cols])   # fit hanya di data latih
X_test_s[num_cols] = scaler.transform(X_test[num_cols])         # data uji hanya transform

smote = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train_s, y_train)  # SMOTE hanya di data latih

print("Sebelum SMOTE:", y_train.value_counts().to_dict())
print("Sesudah SMOTE:", y_train_sm.value_counts().to_dict())

## 5. Pemodelan Regresi Logistik

Dua model dilatih sebagai pembanding: tanpa SMOTE (baseline) dan dengan SMOTE.

In [ ]:
model_base = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
model_base.fit(X_train_s, y_train)

model_smote = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
model_smote.fit(X_train_sm, y_train_sm)

print("Kedua model selesai dilatih.")

## 6. Evaluasi model

In [ ]:
def evaluate(model, X_eval, y_eval, threshold=0.5):
    proba = model.predict_proba(X_eval)[:, 1]
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_eval, pred).ravel()
    return {
        "Accuracy": accuracy_score(y_eval, pred),
        "Precision": precision_score(y_eval, pred, zero_division=0),
        "Recall": recall_score(y_eval, pred),
        "F1": f1_score(y_eval, pred),
        "AUC": roc_auc_score(y_eval, proba),
        "TN": tn, "FP": fp, "FN": fn, "TP": tp,
    }

compare = pd.DataFrame({
    "Tanpa SMOTE": evaluate(model_base, X_test_s, y_test),
    "Dengan SMOTE": evaluate(model_smote, X_test_s, y_test),
}).T
compare.round(3)

In [ ]:
print("=== Classification Report: TANPA SMOTE ===")
print(classification_report(y_test, model_base.predict(X_test_s),
                            target_names=["Tidak Beli", "Beli"], digits=3))
print("=== Classification Report: DENGAN SMOTE ===")
print(classification_report(y_test, model_smote.predict(X_test_s),
                            target_names=["Tidak Beli", "Beli"], digits=3))

**Catatan untuk laporan:** bandingkan Recall kelas "Beli" pada kedua model. 
Jelaskan mengapa akurasi tanpa SMOTE terlihat tinggi tetapi Recall rendah, dan apa konsekuensinya 
(False Negative = pengunjung yang akan membeli tetapi tidak terdeteksi).

### 6.1 Confusion Matrix (kedua model)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, m) in zip(axes, [("Tanpa SMOTE", model_base), ("Dengan SMOTE", model_smote)]):
    cm = confusion_matrix(y_test, m.predict(X_test_s))
    ConfusionMatrixDisplay(cm, display_labels=["Tidak Beli", "Beli"]).plot(
        cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(f"Confusion Matrix: {name}")
plt.tight_layout()
plt.savefig(IMG_DIR / "06_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

### 6.2 Kurva ROC dan AUC

In [ ]:
plt.figure(figsize=(7, 6))
for name, m, color in [("Tanpa SMOTE", model_base, "#5B6B70"), ("Dengan SMOTE", model_smote, "#1F8A70")]:
    proba = m.predict_proba(X_test_s)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    plt.plot(fpr, tpr, color=color, label=f"{name} (AUC = {roc_auc_score(y_test, proba):.3f})")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Tebakan acak")
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR)")
plt.title("Kurva ROC")
plt.legend()
plt.savefig(IMG_DIR / "07_roc_curve.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Analisis threshold

Threshold bawaan 0,5 belum tentu optimal. Menurunkan threshold menaikkan Recall 
(lebih banyak pembeli terdeteksi) dengan konsekuensi lebih banyak False Positive.

In [ ]:
rows = {}
for t in [0.7, 0.6, 0.5, 0.4, 0.3, 0.2]:
    r = evaluate(model_smote, X_test_s, y_test, threshold=t)
    rows[t] = {k: r[k] for k in ["Precision", "Recall", "F1", "FP", "FN"]}
threshold_table = pd.DataFrame(rows).T
threshold_table.index.name = "Threshold"
threshold_table.round(3)

In [ ]:
proba_final = model_smote.predict_proba(X_test_s)[:, 1]
ts = np.linspace(0.05, 0.95, 91)
prec = [precision_score(y_test, (proba_final >= t).astype(int), zero_division=0) for t in ts]
rec = [recall_score(y_test, (proba_final >= t).astype(int)) for t in ts]
f1s = [f1_score(y_test, (proba_final >= t).astype(int)) for t in ts]

plt.figure(figsize=(8, 5))
plt.plot(ts, prec, label="Precision", color="#5B6B70")
plt.plot(ts, rec, label="Recall", color="#1F8A70")
plt.plot(ts, f1s, label="F1-Score", color="#0F2E33", linestyle="--")
plt.xlabel("Threshold")
plt.ylabel("Nilai metrik (kelas Beli)")
plt.title("Trade-off Precision dan Recall terhadap Threshold")
plt.legend()
plt.savefig(IMG_DIR / "08_threshold_tradeoff.png", dpi=150, bbox_inches="tight")
plt.show()

**Pilih threshold final** sesuai strategi bisnis dan jelaskan alasannya di laporan. 
Ubah nilai `THRESHOLD` di bawah (mis. 0.4 jika ingin Recall lebih tinggi). 
Nilai ini otomatis dipakai oleh aplikasi Flask.

In [ ]:
THRESHOLD = 0.5
final_model = model_smote

final_metrics = evaluate(final_model, X_test_s, y_test, threshold=THRESHOLD)
pd.Series(final_metrics).round(3)

## 8. Interpretasi koefisien model

Koefisien positif menaikkan peluang pembelian, koefisien negatif menurunkannya. 
Fitur numerik sudah di-scale, sehingga koefisiennya bisa dibandingkan satu sama lain. 
Kategori yang di-drop (`drop_first=True`) menjadi pembanding: bulan **Aug** dan tipe pengunjung **New_Visitor**.

In [ ]:
coef = pd.Series(final_model.coef_[0], index=X.columns).sort_values()
top = pd.concat([coef.head(10), coef.tail(10)])

plt.figure(figsize=(8, 8))
colors = ["#5B6B70" if v < 0 else "#1F8A70" for v in top.values]
plt.barh(top.index, top.values, color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("10 Koefisien Terendah dan 10 Tertinggi")
plt.xlabel("Koefisien (log-odds)")
plt.savefig(IMG_DIR / "09_koefisien_model.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
odds = pd.DataFrame({"Koefisien": coef, "Odds Ratio": np.exp(coef)}).sort_values("Koefisien", ascending=False)
odds.head(10).round(3)

## 9. Menyimpan model untuk aplikasi Flask

Semua artefak disimpan di folder `model/`. Aplikasi Flask membutuhkan: model, scaler, 
urutan kolom, daftar kolom numerik, nilai default, dan konfigurasi.

In [ ]:
joblib.dump(final_model, MODEL_DIR / "logreg_model.joblib")
joblib.dump(scaler, MODEL_DIR / "scaler.joblib")
joblib.dump(list(X.columns), MODEL_DIR / "columns.joblib")
joblib.dump(num_cols, MODEL_DIR / "num_cols.joblib")

(MODEL_DIR / "defaults.json").write_text(json.dumps(defaults, indent=2))

config = {
    "threshold": THRESHOLD,
    "months": months_present,
    "visitor_types": visitor_types,
    "model_name": "Regresi Logistik + SMOTE",
    "sklearn_version": sklearn.__version__,
    "metrics": {k: round(float(final_metrics[k]), 3) for k in ["Accuracy", "Precision", "Recall", "F1", "AUC"]},
}
(MODEL_DIR / "config.json").write_text(json.dumps(config, indent=2))

print("Tersimpan di", MODEL_DIR.resolve())
print("Versi scikit-learn:", sklearn.__version__, "(gunakan versi ini di requirements.txt)")
for f in sorted(MODEL_DIR.iterdir()):
    print(" -", f.name)